In [1]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import copy
from sklearn.model_selection import train_test_split
from pathlib import Path
import torch.profiler
from enum import Enum
import optuna

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
class ModelType(Enum):
    CNN1D = 1
    CNN_LSTM = 2


In [3]:
data_path = '../../data/'
training_data_path = data_path + "Final Training Data/"

In [4]:
class CNN1D(nn.Module):
    def __init__(self, input_channels, num_classes, activation_fn):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 32, 5, padding=2),
            nn.BatchNorm1d(32),
            activation_fn(),

            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            activation_fn(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128),
            activation_fn(),
            nn.MaxPool1d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            activation_fn(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.mean(dim=2)
        return self.fc(x)

In [19]:
class CNN_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(CNN_LSTM, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out = self.cnn(x)
        
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [7]:
def get_loader(settings, X_train, y_train):
    batch_size = 0
    sampler = None
    shuffle = True
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = 1.0 / class_counts
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        batch_size = 65536
    elif settings["weight"] == "random_weighted":

        loss_fn = torch.nn.CrossEntropyLoss()
        class_counts = Counter(y_train.numpy())
        
        num_samples = len(y_train)

        class_weights = {cls: num_samples / count for cls, count in class_counts.items()}

        sample_weights = [class_weights[label.item()] for label in y_train]
        sample_weights = torch.DoubleTensor(sample_weights)

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        batch_size = 3000
        shuffle = None
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])
        batch_size = settings["batch size"]

    return DataLoader(
                TensorDataset(X_train, y_train),
                batch_size=batch_size,
                shuffle=shuffle,
                sampler=sampler,
                num_workers=1,
                pin_memory=True,
                prefetch_factor=2
            ), loss_fn

In [8]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch, y_batch) in enumerate(loader):
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

In [9]:
def validate_epoch(model, X_val, y_val):
    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    with torch.no_grad():

        X_val_device = X_val.to(device)
        y_val_device = y_val.to(device)

        logits = model(X_val_device)

        probs = torch.softmax(logits, dim=1)

        confidences, preds = torch.max(probs, dim=1)

        mask = confidences >= 0.8

        if mask.sum() == 0:

            val_metric = 0

        else:

            coverage = (
                mask.float().mean().item()
            )

            precision = (
                preds[mask] == y_val_device[mask]
            ).float().mean().item()

            val_metric = (
                precision * coverage
            )

    return val_metric

In [10]:
def train_loso(
    X_train,
    X_test,
    y_train,
    y_test,
    subject_idx,
    settings,
):
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.long, device=device)

    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.long, device=device)

    # ----------------------------
    # validation split
    # ----------------------------
    X_train_np = X_train.cpu().numpy()
    y_train_np = y_train.cpu().numpy()

    X_train_main, X_val, y_train_main, y_val = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.2,
        stratify=y_train_np
    )

    X_train_main = torch.tensor(X_train_main, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)

    y_train_main = torch.tensor(y_train_main, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    model_settings = settings["model settings"]
    training_settings = settings["training settings"]
    
    lr_range = training_settings["learning rate"]
    wd_range = training_settings["weight decay"]

    match model_settings["model"]:
        case ModelType.CNN1D:
            model = CNN1D(
                input_channels=X_train.shape[-1],
                num_classes=len(torch.unique(y_train)),
                activation_fn=model_settings["activation_fn"]
            ).to(device)
        case ModelType.CNN_LSTM:
            hidden_size = training_settings["hidden_size"]
            num_layers = training_settings["num_layers"]
            
            model = CNN_LSTM(
                X_train.shape[-1],
                64,
                1,
                len(torch.unique(y_train))
            ).to(device)
    model = torch.compile(model)
        
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr_range,
        weight_decay=wd_range
    )

    loader, loss_fn = get_loader(training_settings, X_train_main, y_train_main)

    best_val = -1
    best_state = model.state_dict()
    patience_counter = 0

    for epoch in range(training_settings["epoch"]):
        train_epoch(model, loader, optimizer, loss_fn)
        val_score = validate_epoch(model, X_val, y_val)

        if val_score > best_val:
            best_val = val_score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= training_settings["patience"]:
            break
            
    model.load_state_dict(best_state)

    # ----------------------------
    # TEST
    # ----------------------------
    with torch.no_grad():

        model.eval()
        prob = torch.softmax(model(X_test), dim=1).cpu()
        
        confidences, preds = torch.max(prob, dim=1)

        confidences = confidences.numpy()
        preds = preds.cpu()
        y_test_np = y_test.cpu()

        results = []

        for t in settings["training settings"]["thresholds"]:
            mask = confidences >= t

            if mask.sum() == 0:
                results.append((t, 0, 0.0))
                continue

            precision = (preds[mask] == y_test_np[mask]).float().mean().item()

            results.append((t, mask.sum().item(), precision))

        acc = (preds == y_test_np).float().mean().item()

    return acc, results, len(preds)

In [11]:
def load_subject(cache_dir, subject):
    data = np.load(Path(cache_dir) / f"{subject}.npz", allow_pickle=True)
    return data["X"], data["y"]

def get_subjects(cache_dir):
    return sorted([p.stem for p in Path(cache_dir).glob("*.npz")])

In [12]:
def normalize_train_test(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

In [13]:
cache_dir = "../data/Final Training Data/Windowed Data/fa58a95392aed78461eb668748d46ab8"

subjects = get_subjects(cache_dir)

In [18]:
results_log = []
settings = {
    "training settings": {
        "thresholds": [0.7, 0.8, 0.9, 0.95],
        "learning rate": 0.00168,
        "weight decay": 0.00139,

        "patience": 5,
        "epoch": 150,
        
        "weight": None,
        "label smoothing": 0.14,
        "batch size": 32,
            
        "hidden_size": 1024,
        "num_layers": 2,
    },
    "model settings": {
        "model": ModelType.CNN_LSTM,
        "activation_fn": nn.GELU
    }
}

subject_idx = 0
for test_subject in subjects:
    subject_idx += 1

    train_subjects = [s for s in subjects if s != test_subject]

    print(f"\nSubject {subject_idx}: {test_subject}")
    print("Loading data...")

    X_train_list, y_train_list = [], []

    for s in train_subjects:
        X_s, y_s = load_subject(cache_dir, s)
        X_train_list.append(X_s)
        y_train_list.append(y_s)

    X_train = np.concatenate(X_train_list)
    y_train = np.concatenate(y_train_list)

    X_test, y_test = load_subject(cache_dir, test_subject)
    X_train, X_test = normalize_train_test(X_train, X_test)

    print("Training...")

    acc, results, predict_count = train_loso(
        X_train, X_test,
        y_train, y_test,
        subject_idx, settings
    )

    results_log.append({
        "subject": test_subject,
        "acc": acc,
        "results": results,
        "predict_count": predict_count
    })

    print(f"acc: {acc}")
    for i in range(len(results)):
        result = results[i]
        print(f"confidence threshold (%): {result[0] * 100}, coverage: {result[1]}/{predict_count}, of which correct (%) : {result[2] * 100:.4f}")

    print("\n")


Subject 1: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
acc: 0.5639165639877319
confidence threshold (%): 70.0, coverage: 609/1103, of which correct (%) : 65.6814
confidence threshold (%): 80.0, coverage: 221/1103, of which correct (%) : 70.1357
confidence threshold (%): 90.0, coverage: 0/1103, of which correct (%) : 0.0000
confidence threshold (%): 95.0, coverage: 0/1103, of which correct (%) : 0.0000



Subject 2: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
acc: 0.3973063826560974
confidence threshold (%): 70.0, coverage: 475/1782, of which correct (%) : 39.3684
confidence threshold (%): 80.0, coverage: 179/1782, of which correct (%) : 30.1676
confidence threshold (%): 90.0, coverage: 0/1782, of which correct (%) : 0.0000
confidence threshold (%): 95.0, coverage: 0/1782, of which correct (%) : 0.0000



Subject 3: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...


KeyboardInterrupt: 

In [15]:
mean_acc = sum(r["acc"] for r in results_log) / len(results_log)
print(f"mean acc: {mean_acc}")

num_thresholds = len(results_log[0]["results"])
for i in range(num_thresholds):
    threshold = results_log[0]["results"][i][0]

    mean_coverage = (
        sum(
            r["results"][i][1] / r["predict_count"]
            for r in results_log
        )
        / len(results_log)
    )
    
    mean_precision = (
        sum(r["results"][i][2] for r in results_log)
        / len(results_log)
    )

    print(
        f"threshold: {threshold * 100:.2f} %, "
        f"mean coverage: {mean_coverage*100:.2f} % "
        f"mean precision: {mean_precision*100:.4f}"
    )

print("\n")

for i, log in enumerate(results_log):
    print(f"Subject number {i+1}: {log.get('subject')}")
    print(f"acc: {log.get('acc')}")
    for result in log.get("results"):
        print(f"conf threshold (%): {result[0] * 100}, coverage: {result[1]}/{log.get('predict_count')}, correct (%) : {result[2] * 100:.4f}")

    print("\n")

mean acc: 0.5298641459508375
threshold: 70.00 %, mean coverage: 38.71 % mean precision: 60.3487
threshold: 80.00 %, mean coverage: 19.75 % mean precision: 63.3708
threshold: 90.00 %, mean coverage: 1.71 % mean precision: 59.8141
threshold: 95.00 %, mean coverage: 0.02 % mean precision: 9.0909


Subject number 1: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
acc: 0.6418857574462891
conf threshold (%): 70.0, coverage: 530/1103, correct (%) : 76.7925
conf threshold (%): 80.0, coverage: 272/1103, correct (%) : 86.3971
conf threshold (%): 90.0, coverage: 2/1103, correct (%) : 100.0000
conf threshold (%): 95.0, coverage: 0/1103, correct (%) : 0.0000


Subject number 2: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
acc: 0.38439956307411194
conf threshold (%): 70.0, coverage: 592/1782, correct (%) : 38.0068
conf threshold (%): 80.0, coverage: 223/1782, correct (%) : 34.9776
conf threshold (%): 90.0, coverage: 33/1782, correct (%) : 18.1818
conf threshold (%): 95.0, coverage: 1/1782, correct (%) : 0.0000


Subje

In [16]:
def objective(trial):

    settings = {
        "training settings": {
            "thresholds": [0.7, 0.8, 0.9, 0.95],
            "learning rate": trial.suggest_float("lr", 0.0013, 0.0017, log=True),
            "weight decay": trial.suggest_float("wd", 0.0012, 0.0018, log=True),

            "patience": 5,
            "epoch": 150,
        
            "weight": None,
            "label smoothing": trial.suggest_float("ls", 0.0, 1.0),
            "batch size": trial.suggest_categorical("batch", [32, 64, 128, 256, 512]),
            
            "hidden_size": trial.suggest_categorical("h_size", [128, 256, 512, 1024, 2048]),
            "num_layers": trial.suggest_categorical("num_l", [1, 2]),
        },
        "model settings": {
            "model": ModelType.CNN_LSTM,
            "activation_fn": nn.GELU
        }
    }

    # IMPORTANT:
    # You usually DON'T run full LOSO inside HPO
    # Instead use 1–2 validation subjects or subset

    accs = []

    for test_subject in subjects:
        train_subjects = [s for s in subjects if s != test_subject]

        # -------------------
        # LOAD TRAIN
        # -------------------
        X_train_list, y_train_list = [], []

        for s in train_subjects:
            X_s, y_s = load_subject(cache_dir, s)
            X_train_list.append(X_s)
            y_train_list.append(y_s)

        X_train = np.concatenate(X_train_list)
        y_train = np.concatenate(y_train_list)

        # -------------------
        # LOAD TEST
        # -------------------
        X_test, y_test = load_subject(cache_dir, test_subject)

        X_train, X_test = normalize_train_test(X_train, X_test)

        acc, _, _ = train_loso(
            X_train, X_test,
            y_train, y_test,
            0,
            settings
        )

        accs.append(acc)

    return sum(accs) / len(accs)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=45)

print(study.best_params)

[I 2026-07-03 08:29:59,363] A new study created in memory with name: no-name-205cf80a-073b-46a2-812b-0186338be1fb
[W 2026-07-03 08:32:03,372] Trial 0 failed with parameters: {'lr': 0.0016063960706717486, 'wd': 0.0012518294877411373, 'ls': 0.4040621888930793, 'batch': 32, 'h_size': 2048, 'num_l': 2} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_192813/1887766682.py", line 54, in objective
    acc, _, _ = train_loso(
  File "/tmp/ipykernel_192813/700696807.py", line 72, in train_loso
    train_epoch(model, loader, optimizer, loss_fn)
  File "/tmp/ipykernel_192813/2960162716.py", line 6, in train_epoch
    X_batch = X_batch.to(device, non_blocking=True)
KeyboardInterrupt
[W 2026-07-03 08:32:03,375] Trial 0 failed with value None.


KeyboardInterrupt: 